In [ ]:
# Только для разработки: открыть раздел можно из notebooks/ или корня проекта
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'notebook_manifest.json').is_file()
)
_notebook_os.chdir(_notebook_root)


#### 2.2 Проверка корректности данных

##### Дубликаты

Проверяем полные совпадения строк, включая целевую переменную

Отдельно считаем повторные экземпляры сверх первого, все строки в повторяющихся группах и число таких групп

Затем проверяем, встречаются ли одинаковые наборы признаков с разными значениями loan_status.


In [6]:
duplicate_extra = data.duplicated()
duplicate_all = data.duplicated(keep=False)
feature_columns = data.columns.drop('loan_status')
conflicting_groups = (
    data.groupby(list(feature_columns), dropna=False)['loan_status']
        .nunique().gt(1)
)

duplicate_summary = pd.DataFrame({
    'Количество': [
        int(duplicate_extra.sum()),
        int(duplicate_all.sum()),
        len(data.loc[duplicate_all].drop_duplicates()),
        int(conflicting_groups.sum()),
    ]
}, index=[
    'Повторные экземпляры сверх первого',
    'Все строки в повторяющихся группах',
    'Группы полностью одинаковых строк',
    'Группы одинаковых признаков с разными loan_status',
])
display(duplicate_summary)
display(data.loc[duplicate_all].sort_values(list(data.columns)).head(10))


,Количество
Повторные экземпляры сверх первого,165
Все строки в повторяющихся группах,330
Группы полностью одинаковых строк,165
Группы одинаковых признаков с разными loan_status,0


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
15944,21,8088,RENT,NaN,MEDICAL,C,1200,15.23,0,0.15,Y,2
16835,21,8088,RENT,NaN,MEDICAL,C,1200,15.23,0,0.15,Y,2
2431,21,15600,RENT,0,MEDICAL,A,2800,7.4,1,0.18,N,4
17758,21,15600,RENT,0,MEDICAL,A,2800,7.4,1,0.18,N,4
2498,21,18000,RENT,0,DEBTCONSOLIDATION,A,3000,7.9,1,0.17,N,2
17601,21,18000,RENT,0,DEBTCONSOLIDATION,A,3000,7.9,1,0.17,N,2
1778,21,20000,RENT,0,MEDICAL,A,2000,NaN,0,0.1,N,3
17642,21,20000,RENT,0,MEDICAL,A,2000,NaN,0,0.1,N,3
740,21,21600,OWN,NaN,VENTURE,A,7125,6.99,0,0.33,N,3
17283,21,21600,OWN,NaN,VENTURE,A,7125,6.99,0,0.33,N,3


##### Семантические ограничения

Проверяем правдоподобие возраста и согласованность связанных полей

Условия, уже выполненные по доменам из 0.2, повторно не перечисляем

Сначала проверяем верхнюю границу стажа 86 лет, затем согласованность с возрастом только для записей с допустимым стажем

Проверяем ограничения ниже; записи с пропусками в нужных полях учитываем отдельно как не проверенные

| Переменные | Ограничение | Обоснование |
| --- | --- | --- |
| person_age | Возраст ≤ 100 лет | Рабочая верхняя граница для проверки подозрительных значений<br>Минимум 20 лет уже известен и не вызывает вопросов |
| person_emp_length | Стаж ≤ 86 лет | Рабочая верхняя граница: максимальный возраст 100 лет минус минимальный возраст начала работы 14 лет |
| person_emp_length, person_age | Стаж ≤ возраст − 14 | Предполагаем начало работы не раньше 14 лет; это рабочее допущение, а не правило источника<br>При принятой границе возраста максимальный стаж — 86 лет |
| cb_person_cred_hist_length, person_age | Длина кредитной истории ≤ возраст − 16 | Предполагаем начало кредитной истории не раньше 16 лет; это рабочее допущение<br>Оно проверяет согласованность полей, а не сам по себе диапазон истории 2–30 лет |
| loan_percent_income, loan_amnt, person_income | Абсолютная разница между loan_percent_income и loan_amnt / person_income ≤ 0,005 с небольшим запасом на вычисления | Проверяем соответствие доли кредита исходным сумме и доходу с учётом округления до сотых |
| person_age | Возраст — целое число лет | Интерпретируем возраст в полных годах |
| person_emp_length | Стаж — целое число лет | Проверяем предположение о записи стажа в полных годах; дробный стаж сам по себе возможен |
| cb_person_cred_hist_length | Длина кредитной истории — целое число лет | Проверяем представление длительности в полных годах |


In [7]:
ratio_expected = data['loan_amnt'] / data['person_income']
ratio_difference = (data['loan_percent_income'] - ratio_expected).abs()

checks = {
    'Целочисленность возраста': (
        ['person_age'], data['person_age'] % 1 != 0),
    'Целочисленность стажа': (
        ['person_emp_length'], data['person_emp_length'] % 1 != 0),
    'Целочисленность кредитной истории': (
        ['cb_person_cred_hist_length'], data['cb_person_cred_hist_length'] % 1 != 0),
    'Возраст ≤ 100': (
        ['person_age'], data['person_age'] > 100),
    'Стаж ≤ 86': (
        ['person_emp_length'], data['person_emp_length'] > 86),
    'Стаж ≤ возраст − 14': (
        ['person_age', 'person_emp_length'],
        data['person_emp_length'] > data['person_age'] - 14),
    'Кредитная история ≤ возраст − 16': (
        ['person_age', 'cb_person_cred_hist_length'],
        data['cb_person_cred_hist_length'] > data['person_age'] - 16),
    'Доля кредита соответствует сумме / доходу (погрешность ≤ 0,005)': (
        ['loan_percent_income', 'loan_amnt', 'person_income'],
        ratio_difference > 0.005 + 1e-9),
}

constraint_results = []
constraint_violations = pd.DataFrame(index=data.index)
for name, (columns, violation) in checks.items():
    available = data[columns].notna().all(axis=1)
    # Согласованность проверяем только при допустимом абсолютном стаже.
    if name == 'Стаж ≤ возраст − 14':
        available &= data['person_emp_length'] <= 86
    if 'person_income' in columns:
        available &= data['person_income'] > 0
    constraint_violations[name] = available & violation
    constraint_results.append({
        'Проверка': name,
        'Проверено': int(available.sum()),
        'Не проверено': int((~available).sum()),
        'Нарушений': int(constraint_violations[name].sum()),
    })

display(pd.DataFrame(constraint_results).set_index('Проверка').rename_axis(None))


,Проверено,Не проверено,Нарушений
Целочисленность возраста,32581,0,0
Целочисленность стажа,31686,895,0
Целочисленность кредитной истории,32581,0,0
Возраст ≤ 100,32581,0,5
Стаж ≤ 86,31686,895,2
Стаж ≤ возраст − 14,31684,897,0
Кредитная история ≤ возраст − 16,32581,0,0
"Доля кредита соответствует сумме / доходу (погрешность ≤ 0,005)",32581,0,429


#### Выводы и решения после 2.2

<table>
<thead><tr><th>Выводы</th><th>Решения</th></tr></thead>
<tbody>
<tr><td>В датасете 32 581 запись, 11 признаков и одна целевая переменная</td><td>—</td></tr>
<tr><td>Среди признаков: 7 числовых, 2 номинальных, 1 порядковый и 1 бинарный</td><td>—</td></tr>
<tr><td>В person_emp_length (2,75 % записей) и в loan_int_rate (9.56% записей) есть пропуски</td><td>Способ обработки пропусков выберем после анализа распределений</td></tr>
<tr><td>Целевая переменная заполнена во всех записях и содержит только значения 0 и 1</td><td>—</td></tr>
<tr><td style="background-color: #f8dddd; color: #6b1c1c;">− Наблюдаемые максимумы возраста (144 года) и стажа (123 года) – явные выбросы, а конкретно ошибки данных</td><td style="background-color: #f8dddd; color: #6b1c1c;">Заменён уточнёнными выводами о возрасте и стаже ниже</td></tr>
<tr><td>Числовые признаки имеют существенно разные масштабы: максимум дохода — 6 000 000, максимум доли кредита от дохода — 0,83</td><td>—</td></tr>
<tr><td style="background-color: #dcefe3; color: #172b20;">+ Найдено 165 групп ровно по две полностью одинаковые строки: 330 записей, из них 165 повторных (0,51 % выборки)<br>Такая структура согласуется с дублированием при вводе или сборке данных</td><td style="background-color: #dcefe3; color: #172b20;">Считать повторы техническими дубликатами и удалить, оставив по одной записи в каждой группе</td></tr>
<tr><td style="background-color: #dcefe3; color: #172b20;">+ Возраст превышает рабочую границу 100 лет в 5 записях: 144 года в трёх записях, 123 года в двух</td><td style="background-color: #dcefe3; color: #172b20;">Удалить эти 5 записей<br>Обработка неопределенных значений для такой малой части выборки не стоит свеч</td></tr>
<tr><td style="background-color: #dcefe3; color: #172b20;">+ Верхнюю границу стажа 86 лет превышают 2 записи: в обеих стаж составляет 123 года</td><td style="background-color: #dcefe3; color: #172b20;">Заменить недопустимый стаж пропусками, сохранив записи</td></tr>
<tr><td style="background-color: #dcefe3; color: #172b20;">+ Для записей со стажем ≤ 86 лет выполняется условие «стаж ≤ возраст − 14»</td><td style="background-color: #dcefe3; color: #172b20;">—</td></tr>
<tr><td style="background-color: #dcefe3; color: #172b20;">+ Выполняется условие «кредитная история ≤ возраст − 16»</td><td style="background-color: #dcefe3; color: #172b20;">—</td></tr>
<tr><td style="background-color: #dcefe3; color: #172b20;">+ В 429 записях (1,32 % всей выборки) доля кредита отличается от отношения суммы к доходу более чем на 0,005; одним округлением до сотых эти расхождения не объясняются</td><td style="background-color: #dcefe3; color: #172b20;">—</td></tr>
<tr><td style="background-color: #dcefe3; color: #172b20;">+ Все заполненные значения возраста, стажа и длины кредитной истории целочисленные</td><td style="background-color: #dcefe3; color: #172b20;">—</td></tr>
</tbody>
</table>


##### Промежуточная очистка

Применяем принятые решения перед продолжением разведочного анализа

Исходная таблица сохранена в `data_raw`, дальнейший анализ ведём на `data`

Пропуски стажа пока не заполняем


In [ ]:
# Каждый запуск повторяет очистку исходных данных
data = data_raw.drop_duplicates().copy()
invalid_age = data['person_age'] > 100
data = data.loc[~invalid_age].copy()

invalid_employment = data['person_emp_length'] > 86
data.loc[invalid_employment, 'person_emp_length'] = float('nan')
